# RT-DETRv3-R18: sondeo de velocidad (iteración 1)

Entrena unos minutos con cuatro configuraciones y mide, en cada una, el tiempo por paso, cuánto de ese tiempo es espera de datos, la memoria de la GPU y si aparecen NaN al principio. Con eso se estima cuánto tardarían 100 épocas y se elige la configuración de la corrida completa. **No guarda ningún modelo y no usa validación ni prueba**: ninguna época llega a evaluarse.

| Sondeo | GPU | Lote por GPU | Lote total | lr | Precisión |
|---|---|---|---|---|---|
| `1gpu_fp32` | 1 | 4 | 4 | 0.000025 | completa (receta oficial) |
| `2gpu_fp32` | 2 | 4 | 8 | 0.00005 | completa, BatchNorm sincronizada entre GPUs (oficial con varias GPU) |
| `1gpu_amp` | 1 | 4 | 4 | 0.000025 | mixta (como YOLO26) |
| `2gpu_amp` | 2 | 4 | 8 | 0.00005 | mixta |

La tasa de aprendizaje es la oficial (0.0004 para un lote total de 64) escalada al lote total.

Usa la versión 2 del parche de lotes sin cajas: esos lotes ya no se saltan, se entrenan sin la parte de denoising, como hace el RT-DETR original. Probado en CPU con un proceso y con dos (`docs/modelado`, conversación del 5 de octubre).

Configuración del cuaderno: acelerador **GPU T4 x2**, **Internet activado**, dataset `deteccionoso` adjunto. Lanzar con *Save Version > Save & Run All (Commit)*: tarda unos 35 minutos y el resumen queda en la salida de la versión.

## 0. Parámetros

In [ ]:
REPO    = "https://github.com/clxia12/RT-DETRv3.git"
COMMIT  = "349e7d99a5065e7b684118912e6a74178d4f4625"     # último commit, verificado el 3 oct 2026
DATASET = "/kaggle/input/datasets/josemartinvallejos/deteccionoso"
ID_DRIVE_PESOS = "1zIDOjn1qDccC3TBsDlGQHOjVrehd26bk"     # RT-DETRv3-R18 COCO (Google Drive del README; el enlace de Baidu da 404)
MD5_PESOS      = "25cdd032d57dbe7773753ae65c300c63"      # medido el 3 oct 2026

MINUTOS_POR_SONDEO = 6
SONDEOS = [  # nombre, número de GPU, lote por GPU, precisión mixta
    ("1gpu_fp32", 1, 4, False),
    ("2gpu_fp32", 2, 4, False),
    ("1gpu_amp",  1, 4, True),
    ("2gpu_amp",  2, 4, True),
]
SEG_EVAL_POR_EPOCA = 40     # validación por época medida el 5 oct (598 fotos, unos 35-40 s); solo para la estimación
LIMITE_HORAS = 11.0         # para que 100 épocas quepan en una sesión de Kaggle (12 h) con margen
ORDEN_FIDELIDAD = ["2gpu_fp32", "1gpu_fp32", "2gpu_amp", "1gpu_amp"]   # de más a menos fiel a la receta oficial

## 1. Entorno: Python 3.12 aparte, con Paddle 3.3.0 y las versiones probadas

In [ ]:
import os, sys, subprocess, hashlib
from pathlib import Path

TRABAJO = Path("/kaggle/working")
VENV    = "/tmp/venv_paddle"          # fuera de /kaggle/working: no se guarda con la versión
VENV_PY = f"{VENV}/bin/python"

def correr(cmd, ver=3000, env=None):
    """Corre un comando y muestra el final de su salida y el código de salida."""
    print("$", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
                       env={**os.environ, **(env or {})})
    if r.stdout.strip():
        print(r.stdout[-ver:])
    if r.stderr.strip():
        print("[stderr]", r.stderr[-ver:])
    print("-> código de salida:", r.returncode, "\n")
    return r

REQUISITOS = """
anyio==4.15.1 babel==2.18.0 bce-python-sdk==0.9.79 blinker==1.9.0 certifi==2026.7.22
charset-normalizer==3.5.2 click==8.5.0 cloudpickle==3.1.2 contourpy==1.3.3 crc32c==2.9.post0
cycler==0.12.1 cython==3.3.0 flask==3.1.3 flask-babel==4.0.0 fonttools==4.66.1 future==1.0.0
h11==0.16.0 httpcore==1.0.9 httpx==0.28.1 idna==3.20 imageio==2.38.0 imgaug==0.4.0
itsdangerous==2.2.0 jinja2==3.1.6 joblib==1.6.0 kiwisolver==1.5.1 lapx==0.10.0 lazy-loader==0.6
markupsafe==3.0.4 matplotlib==3.11.2 motmetrics==1.4.0 narwhals==2.26.0 networkx==3.7
numpy==1.26.4 opencv-python==4.5.5.64 opt-einsum==3.3.0 packaging==26.3 pandas==3.0.6
pillow==12.3.0 protobuf==7.36.2 psutil==7.2.2 pyclipper==1.4.0 pycocotools==2.0.11
pycryptodome==3.23.0 pyparsing==3.3.3 python-dateutil==2.9.0.post0 pytz==2026.5 pyyaml==6.0.3
rarfile==4.5 requests==2.34.2 safetensors==0.8.0 scikit-image==0.26.0 scikit-learn==1.9.1
scipy==1.17.1 setuptools==80.10.2 shapely==2.1.2 six==1.17.0 terminaltables==3.1.10
threadpoolctl==3.7.0 tifffile==2026.3.3 tqdm==4.70.1 typeguard==4.6.0 typing-extensions==4.16.0
urllib3==2.8.0 visualdl==2.5.3 werkzeug==3.1.9 xmltodict==1.0.4
""".split()
(TRABAJO / "requisitos_rtdetrv3.txt").write_text("\n".join(REQUISITOS) + "\n")
print(len(REQUISITOS), "paquetes fijados")   # 67

correr("nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv")
print("CPU que ve la sesión:", os.cpu_count())
correr([sys.executable, "-m", "pip", "install", "-q", "uv"])
correr([sys.executable, "-m", "uv", "python", "install", "3.12.13"])
correr([sys.executable, "-m", "uv", "venv", "--seed", "--clear", "--python", "3.12.13", VENV])
correr([VENV_PY, "-m", "pip", "install", "-q", "-r", str(TRABAJO / "requisitos_rtdetrv3.txt")])
correr([VENV_PY, "-m", "pip", "install", "-q", "paddlepaddle-gpu==3.3.0",
        "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "--extra-index-url", "https://pypi.org/simple"])
correr([VENV_PY, "-m", "pip", "check"])
correr([VENV_PY, "-c", "import paddle; paddle.utils.run_check()"], env={"CUDA_VISIBLE_DEVICES": "0,1"})

## 2. Repositorio oficial, pesos COCO y parche

In [ ]:
REPO_DIR = TRABAJO / "RT-DETRv3"
if not REPO_DIR.exists():
    correr(["git", "clone", "-q", REPO, str(REPO_DIR)])
correr(["git", "-C", str(REPO_DIR), "checkout", "-q", COMMIT])
correr(["git", "-C", str(REPO_DIR), "log", "-1", "--format=%H %ci %s"])

PESOS_COCO = TRABAJO / "pesos" / "rtdetrv3_r18vd_6x_coco.pdparams"
PESOS_COCO.parent.mkdir(exist_ok=True)
if not PESOS_COCO.exists():
    correr([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    correr(["gdown", ID_DRIVE_PESOS, "-O", str(PESOS_COCO)])
md5 = hashlib.md5(PESOS_COCO.read_bytes()).hexdigest()
print(PESOS_COCO.name, f"{PESOS_COCO.stat().st_size / 1e6:.1f} MB", "md5", md5)
assert md5 == MD5_PESOS, "Los pesos descargados no son los esperados"

In [ ]:
%%writefile /kaggle/working/parche_lotes_sin_cajas_v2.py
"""Parche del proyecto oso andino (versión 2, 2026-10-05) al código oficial de RT-DETRv3.

Problema: si ninguna imagen de un lote de entrenamiento tiene cajas, la función de denoising devuelve None
para todos los grupos de consultas y el código de RT-DETRv3 falla al unir piezas (paddle.concat con None).
El RT-DETR original sí contempla ese caso (comprueba None); la versión 3, con varios grupos, no.

Este parche hace que esos lotes se entrenen como en el RT-DETR original: sin consultas de denoising, solo con
las consultas normales, cuya pérdida de clasificación enseña al modelo que en esas fotos no hay oso.
Reemplaza al parche de la versión 1, que saltaba esos lotes (y que con 2 GPU congelaría el entrenamiento).

Toca tres sitios:
  1. rtdetr_transformerv3.py, _get_decoder_input: no concatena el denoising de un grupo si es None.
  2. rtdetr_transformerv3.py, forward: arma la máscara de atención de un grupo sin denoising y le pone un
     dn_meta de marcador con el número de consultas del grupo, para que la cabeza pueda separar los grupos.
  3. detr_head.py, DINOv3Head.forward: en los grupos marcados calcula la pérdida sin denoising y repite la
     reducción entre GPUs que la pérdida de denoising habría hecho, para que todas las GPUs hagan las mismas.

Uso: python parche_lotes_sin_cajas_v2.py RUTA_DEL_REPOSITORIO   (se puede correr varias veces)"""
import sys
from pathlib import Path

repo = Path(sys.argv[1])
MARCA = "Parche del proyecto oso andino v2"

CAMBIOS = {
    repo / "ppdet/modeling/transformers/rtdetr_transformerv3.py": [
        (  # 1a. referencia de las cajas de denoising
            "            if denoising_bbox_unacts is not None and not (self.o2m_branch and g_id == self.num_groups - 1):\n",
            "            if denoising_bbox_unacts is not None and not (self.o2m_branch and g_id == self.num_groups - 1) \\\n"
            "                    and denoising_bbox_unacts[g_id] is not None:  # " + MARCA + "\n",
        ),
        (  # 1b. consultas de denoising
            "            if denoising_classes is not None and not (self.o2m_branch and g_id == self.num_groups - 1):\n",
            "            if denoising_classes is not None and not (self.o2m_branch and g_id == self.num_groups - 1) \\\n"
            "                    and denoising_classes[g_id] is not None:  # " + MARCA + "\n",
        ),
        (  # 2a. máscara de atención de un grupo sin denoising
            "                else:\n"
            "                    end = end + attn_masks[g_id].shape[1]\n",
            "                elif attn_masks[g_id] is None:  # " + MARCA + ": lote sin cajas, grupo sin denoising\n"
            "                    end = end + self.num_queries[g_id]\n"
            "                    new_mask = new_mask > 0.1 if g_id > 0 else new_mask >= 0.0\n"
            "                    new_attn_mask[begin: end, begin: end] = new_mask\n"
            "                else:\n"
            "                    end = end + attn_masks[g_id].shape[1]\n",
        ),
        (  # 2b. marcador para la cabeza
            "            attn_masks = new_attn_mask\n",
            "            attn_masks = new_attn_mask\n"
            "            # " + MARCA + ": los grupos sin denoising llevan un dn_meta de marcador con su número de consultas\n"
            "            dn_metas = [dm if dm is not None else {'dn_num_split': [0, self.num_queries[i]], 'sin_denoising': True}\n"
            "                        for i, dm in enumerate(dn_metas)]\n",
        ),
    ],
    repo / "ppdet/modeling/heads/detr_head.py": [
        (  # 3a. separación por grupo
            "                for g_id in range(num_groups):\n"
            "                    dn_out_bboxes_gid, dec_out_bboxes_gid = paddle.split(\n"
            "                        dec_out_bboxes[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n"
            "                    dn_out_logits_gid, dec_out_logits_gid = paddle.split(\n"
            "                        dec_out_logits[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n",
            "                for g_id in range(num_groups):\n"
            "                    sin_dn = dn_meta[g_id].get('sin_denoising', False)  # " + MARCA + "\n"
            "                    if sin_dn:\n"
            "                        dn_out_bboxes_gid = dn_out_logits_gid = None\n"
            "                        dec_out_bboxes_gid, dec_out_logits_gid = dec_out_bboxes[g_id], dec_out_logits[g_id]\n"
            "                    else:\n"
            "                        dn_out_bboxes_gid, dec_out_bboxes_gid = paddle.split(\n"
            "                            dec_out_bboxes[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n"
            "                        dn_out_logits_gid, dec_out_logits_gid = paddle.split(\n"
            "                            dec_out_logits[g_id], dn_meta[g_id]['dn_num_split'], axis=2)\n",
        ),
        (  # 3b. pérdida sin denoising y reducción entre GPUs equivalente
            "                        dn_out_logits=dn_out_logits_gid,\n"
            "                        dn_meta=dn_meta[g_id])\n",
            "                        dn_out_logits=dn_out_logits_gid,\n"
            "                        dn_meta=None if sin_dn else dn_meta[g_id])\n"
            "                    if sin_dn:  # " + MARCA + ": misma cantidad de reducciones entre GPUs que con denoising\n"
            "                        self.loss._get_num_gts(inputs['gt_class'])\n",
        ),
    ],
}

for ruta, cambios in CAMBIOS.items():
    texto = ruta.read_text()
    if MARCA in texto:
        print("Ya estaba aplicado en", ruta.name)
        continue
    for viejo, nuevo in cambios:
        n = texto.count(viejo)
        if n != 1:
            raise SystemExit(f"{ruta.name}: el texto de anclaje aparece {n} veces (se esperaba 1); no se parcha nada.")
        texto = texto.replace(viejo, nuevo)
    ruta.write_text(texto)
    print("Parche aplicado en", ruta.name)

In [ ]:
%%writefile /kaggle/working/entrenar_con_semilla.py
"""Lanza tools/train.py del repositorio oficial de RT-DETRv3 fijando antes la semilla.
El repositorio solo permite la semilla 0 (opción --enable_ce); esto llama a la misma función
(ppdet.engine.set_random_seed) con la semilla que se elija, sin modificar el código oficial.
Uso: python entrenar_con_semilla.py RUTA_REPO SEMILLA [argumentos de tools/train.py ...]"""
import runpy
import sys

repo, semilla, resto = sys.argv[1], int(sys.argv[2]), sys.argv[3:]
if "--enable_ce" in resto:
    raise SystemExit("No uses --enable_ce aquí: volvería a fijar la semilla en 0.")
sys.path.insert(0, repo)
from ppdet.engine import set_random_seed
set_random_seed(semilla)
print(f"Semilla fijada en {semilla} (paddle, random y numpy)", flush=True)
sys.argv = [f"{repo}/tools/train.py"] + resto
runpy.run_path(f"{repo}/tools/train.py", run_name="__main__")

In [ ]:
correr([VENV_PY, str(TRABAJO / "parche_lotes_sin_cajas_v2.py"), str(REPO_DIR)])
correr(["git", "-C", str(REPO_DIR), "diff", "--stat"])

## 3. Anotaciones COCO y configuración

In [ ]:
import json
from PIL import Image

COCO_DIR = TRABAJO / "coco"
GIRA = {5, 6, 7, 8}   # orientaciones EXIF con las que OpenCV gira la foto e intercambia ancho y alto

def yolo_a_coco(dir_imagenes, dir_etiquetas, salida_json, nombre_clase="oso"):
    """Etiquetas YOLO de una clase -> JSON COCO. Entran todas las imágenes; las negativas, sin anotaciones."""
    imagenes, anotaciones = [], []
    n_pos = n_neg = n_recortadas = n_descartadas = n_exif = 0
    for img_id, ruta in enumerate(sorted(Path(dir_imagenes).glob("*.jpg")), start=1):
        with Image.open(ruta) as im:
            W, H = im.size                      # solo lee la cabecera
            orient = im.getexif().get(0x0112, 1)
        if orient in GIRA:                      # PaddleDetection (cv2.imdecode) y Ultralytics (cv2.imread) aplican el giro
            W, H = H, W
        n_exif += orient != 1
        imagenes.append({"id": img_id, "file_name": ruta.name, "width": W, "height": H})
        txt = Path(dir_etiquetas) / (ruta.stem + ".txt")
        filas = [l.split() for l in txt.read_text().splitlines() if l.strip()] if txt.exists() else []
        cajas_img = 0
        for f in filas:
            cx, cy, w, h = (float(v) for v in f[1:5])
            x1, y1, x2, y2 = (cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H
            r = (max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2))
            n_recortadas += r != (x1, y1, x2, y2)
            x1, y1, x2, y2 = r
            if x2 - x1 <= 0 or y2 - y1 <= 0:
                n_descartadas += 1
                continue
            anotaciones.append({"id": len(anotaciones) + 1, "image_id": img_id, "category_id": 1,
                                "bbox": [x1, y1, x2 - x1, y2 - y1], "area": (x2 - x1) * (y2 - y1), "iscrowd": 0})
            cajas_img += 1
        n_pos += cajas_img > 0
        n_neg += cajas_img == 0
    Path(salida_json).parent.mkdir(parents=True, exist_ok=True)
    Path(salida_json).write_text(json.dumps({"images": imagenes, "annotations": anotaciones,
                                             "categories": [{"id": 1, "name": nombre_clase}]}))
    return {"imagenes": len(imagenes), "con_oso": n_pos, "sin_oso": n_neg, "cajas": len(anotaciones),
            "cajas_recortadas_al_borde": n_recortadas, "cajas_descartadas": n_descartadas,
            "con_orientacion_exif": n_exif}

ESPERADO = {"train": {"imagenes": 2656, "con_oso": 1382, "cajas": 1507},   # informe de preparación
            "val":   {"imagenes": 598,  "con_oso": 326,  "cajas": 351}}
for conj in ("train", "val"):
    res = yolo_a_coco(Path(DATASET) / "images" / conj, Path(DATASET) / "labels" / conj, COCO_DIR / f"{conj}.json")
    print(conj, res)
    for k, v in ESPERADO[conj].items():
        assert res[k] == v, f"{conj}: {k} = {res[k]}, se esperaba {v}"
print("Conteos iguales a los del informe de preparación.\n")

CONFIG = REPO_DIR / "configs" / "rtdetrv3" / "rtdetrv3_r18vd_oso.yml"
CONFIG.write_text(f"""# RT-DETRv3-R18 para la detección de oso (iteración 1, dataset proxy). Hereda la receta oficial.
_BASE_: [
  'rtdetrv3_r18vd_6x_coco.yml',
]

num_classes: 1
metric: COCO

TrainDataset:
  name: COCODataSet
  dataset_dir: {DATASET}
  image_dir: images/train
  anno_path: {COCO_DIR / 'train.json'}
  data_fields: ['image', 'gt_bbox', 'gt_class', 'is_crowd']
  allow_empty: true            # sin esto se descartan las fotos sin oso

EvalDataset:
  name: COCODataSet
  dataset_dir: {DATASET}
  image_dir: images/val
  anno_path: {COCO_DIR / 'val.json'}
  allow_empty: true

TestDataset:
  name: ImageFolder
  anno_path: {COCO_DIR / 'val.json'}

epoch: 100                     # protocolo común con YOLO26 (la receta oficial usa 72)
LearningRate:
  base_lr: 0.000025            # oficial: 0.0004 para lote total 64; escalada al lote total (aquí, 4)
TrainReader:
  batch_size: 4                # por GPU; con 8 y 16 no cabe en una T4 de 15 GB
  use_shared_memory: false     # con memoria compartida, las fotos sin oso tumban el lector de datos (Paddle 3.3.0)
pretrain_weights: {PESOS_COCO}
""")
print(CONFIG.read_text())

## 4. Sondeos

Cada sondeo entrena `MINUTOS_POR_SONDEO` minutos y se detiene. Del registro de PaddleDetection toma, por paso de entrenamiento, el tiempo total (`batch_cost`) y la espera de datos (`data_cost`), descartando los dos primeros registros (arranque). Mide la memoria con `nvidia-smi` cada 15 segundos.

Con dos GPU se usa `paddle.distributed.launch`, que reparte cada lote entre las dos GPU, y la configuración oficial convierte la BatchNorm en BatchNorm sincronizada (`norm_type: sync_bn`).

In [ ]:
import re, signal, statistics, time, shutil

SONDEO_DIR = TRABAJO / "sondeos"
SONDEO_DIR.mkdir(exist_ok=True)
LANZADOR = TRABAJO / "entrenar_con_semilla.py"

def memoria_gpu():
    q = subprocess.run("nvidia-smi --query-gpu=index,memory.used --format=csv,noheader,nounits",
                       shell=True, capture_output=True, text=True)
    if q.returncode != 0 or not q.stdout.strip():
        return {}
    return {int(a): int(b) for a, b in (l.split(",") for l in q.stdout.strip().splitlines())}

def sondear(nombre, n_gpu, lote, amp, minutos):
    salida = SONDEO_DIR / nombre
    shutil.rmtree(salida, ignore_errors=True)
    salida.mkdir(parents=True)
    lr = 0.0004 * lote * n_gpu / 64
    opciones = (["--amp"] if amp else []) + ["-c", str(CONFIG), "-o", f"save_dir={salida}", "log_iter=10",
                                             f"TrainReader.batch_size={lote}", f"LearningRate.base_lr={lr:.8f}"]  # con decimales fijos: "5e-05" se leería como texto
    dispositivos = ",".join(str(i) for i in range(n_gpu))
    if n_gpu == 1:
        cmd = [VENV_PY, str(LANZADOR), str(REPO_DIR), "0", *opciones]
        registro = salida / "consola.log"
    else:
        cmd = [VENV_PY, "-m", "paddle.distributed.launch", "--devices", dispositivos, "--log_dir", str(salida / "log"),
               str(LANZADOR), str(REPO_DIR), "0", *opciones]
        registro = salida / "log" / "workerlog.0"
    print(f"=== {nombre}: {n_gpu} GPU, lote {lote} por GPU, lr {lr:g}, {'mixta' if amp else 'completa'}")
    print("$", " ".join(cmd), flush=True)
    mem, inicio = {}, time.time()
    with open(salida / "consola.log", "w") as f:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=f, stderr=subprocess.STDOUT, start_new_session=True,
                                env={**os.environ, "CUDA_VISIBLE_DEVICES": dispositivos, "MPLBACKEND": "Agg"})
        while proc.poll() is None and time.time() - inicio < minutos * 60:
            time.sleep(15)
            for k, v in memoria_gpu().items():
                mem[k] = max(mem.get(k, 0), v)
        termino_solo = proc.poll() is not None
        ultimo = registro.stat().st_mtime if registro.exists() else inicio
        sin_avance = (not termino_solo) and time.time() - ultimo > 120   # el registro no cambió en 2 minutos
        if not termino_solo:                       # se detiene el sondeo (y sus procesos hijos)
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=60)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
    time.sleep(10)                                  # que la GPU libere la memoria antes del siguiente
    for p in salida.glob("*.pd*"):                  # si alguna época llegó a guardarse, no hace falta
        p.unlink()

    texto = registro.read_text(errors="replace") if registro.exists() else ""
    consola = (salida / "consola.log").read_text(errors="replace")
    pasos = []
    for linea in texto.splitlines():
        if "Epoch: [" not in linea or "batch_cost" not in linea:
            continue
        m = re.search(r"Epoch: \[(\d+)\] \[\s*(\d+)/(\d+)\]", linea)
        bc = re.search(r"batch_cost: ([\d.]+)", linea)
        dc = re.search(r"data_cost: ([\d.]+)", linea)
        ls = re.search(r" loss: (\S+)", linea)
        if m and bc and dc:
            pasos.append({"epoca": int(m.group(1)), "paso": int(m.group(2)), "pasos_epoca": int(m.group(3)),
                          "batch_cost": float(bc.group(1)), "data_cost": float(dc.group(1)),
                          "loss": ls.group(1) if ls else ""})
    sin_memoria = "out of memory" in (texto + consola).lower()
    hay_nan = any("nan" in p["loss"].lower() for p in pasos)
    utiles = pasos[2:]
    fila = {"sondeo": nombre, "gpu": n_gpu, "lote_total": lote * n_gpu, "precision": "mixta" if amp else "completa",
            "registros": len(pasos), "nan": hay_nan,
            "memoria_max_MiB": " / ".join(f"GPU{k}: {v}" for k, v in sorted(mem.items()))}
    if termino_solo:
        fila["estado"] = "sin memoria" if sin_memoria else f"falló (código {proc.returncode}); ver {salida.name}/consola.log"
    elif sin_avance:
        fila["estado"] = "sin avance en los últimos 2 min (¿bloqueado?)"
    elif len(utiles) < 5:
        fila["estado"] = "pocos registros: no se puede medir"
    else:
        fila["estado"] = "ok"
    if utiles:
        bc = statistics.median(p["batch_cost"] for p in utiles)
        dc = statistics.median(p["data_cost"] for p in utiles)
        pe = utiles[-1]["pasos_epoca"]
        min_epoca = (pe * bc + SEG_EVAL_POR_EPOCA) / 60
        fila.update({"pasos_por_epoca": pe, "s_por_paso": round(bc, 3), "espera_datos_pct": round(100 * dc / bc),
                     "img_por_s": round(lote * n_gpu / bc, 1), "min_por_epoca": round(min_epoca, 1),
                     "horas_100_epocas": round(min_epoca * 100 / 60, 1)})
    print({k: v for k, v in fila.items() if k != "sondeo"}, "\n", flush=True)
    return fila

RESULTADOS = [sondear(n, g, l, a, MINUTOS_POR_SONDEO) for n, g, l, a in SONDEOS]

## 5. Resumen y propuesta

In [ ]:
import pandas as pd

res = pd.DataFrame(RESULTADOS)
pd.set_option("display.width", 250)
cols = [c for c in ["sondeo", "estado", "gpu", "lote_total", "precision", "pasos_por_epoca", "s_por_paso",
                    "espera_datos_pct", "img_por_s", "min_por_epoca", "horas_100_epocas", "nan", "memoria_max_MiB"]
        if c in res.columns]
print(res[cols].to_string(index=False))
res.to_csv(TRABAJO / "sondeo_rtdetrv3.csv", index=False)

validos = res[(res.estado == "ok") & (~res.nan)] if "horas_100_epocas" in res.columns else res.iloc[0:0]
elegido = None
for nombre in ORDEN_FIDELIDAD:
    f = validos[validos.sondeo == nombre]
    if len(f) and f.horas_100_epocas.iloc[0] <= LIMITE_HORAS:
        elegido = nombre
        break
print()
if elegido:
    print(f"Propuesta según la regla acordada: {elegido} (la más fiel a la receta oficial que cabe en {LIMITE_HORAS:g} h).")
elif len(validos):
    rapido = validos.sort_values("horas_100_epocas").iloc[0]
    print(f"Ninguna cabe en {LIMITE_HORAS:g} h. La más rápida es {rapido.sondeo} ({rapido.horas_100_epocas} h): "
          "habría que repartir la corrida en dos sesiones.")
else:
    print("Ningún sondeo dio datos válidos: revisar los consola.log.")
print("Los NaN solo se vigilan durante los minutos del sondeo; en la corrida completa se siguen vigilando.")

for p in SONDEO_DIR.rglob("*.pd*"):
    p.unlink()
shutil.make_archive(str(TRABAJO / "sondeo_registros"), "zip", SONDEO_DIR)
print("Registros guardados en sondeo_registros.zip y la tabla en sondeo_rtdetrv3.csv")